# 02 — Controlled attack baselines

Every action preserves image size, RGB channels, and the valid pixel range. Strength is normalized to `[0, 1]`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ppo.actions import ACTION_NAMES, apply_action

height, width = 160, 224
x = np.linspace(32, 240, width, dtype=np.uint8)
gradient = np.tile(x, (height, 1))
checker = ((np.indices((height, width)).sum(axis=0) // 10) % 2 * 20).astype(np.uint8)
image = Image.fromarray(np.stack([gradient, np.clip(gradient + checker, 0, 255), gradient], axis=2)).convert("RGB")

In [ ]:
strength = 0.75
outputs = {
    name: apply_action(image, action_id, strength, np.random.default_rng(42))
    for action_id, name in ACTION_NAMES.items()
}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for axis, (name, output) in zip(axes.flat, outputs.items()):
    axis.imshow(output)
    axis.set_title(name.replace("_", " ").title())
    axis.axis("off")
axes.flat[-1].axis("off")
plt.tight_layout()

In [ ]:
for name, output in outputs.items():
    pixels = np.asarray(output)
    assert output.mode == "RGB" and output.size == image.size
    assert pixels.dtype == np.uint8 and 0 <= pixels.min() <= pixels.max() <= 255
    difference = np.abs(pixels.astype(float) - np.asarray(image).astype(float)).mean()
    print(f"{name:20s} mean absolute pixel change={difference:.3f}")